In [15]:
from typing import NamedTuple
from jaxtyping import Float, Array
from typing import List
import jax 
import jax.numpy as jnp 

In [16]:
class SimpleMDP(NamedTuple):
    """
    Finite MDP with deterministic rewards. 
    """
    S: int # number of states 
    A: int # number of states 
    mu: Float[Array, "S"] # Initial state distriution 
    P: Float[Array, "S A S"] # Transition matrix: state x action -> next state 
    r: Float[Array, "S A"] # Reward function 
    H: int # Horizon 

class Transition(NamedTuple):
    """
    One transition consists of a single state-action-reward tuple
    """
    s: int
    a: int
    r: float  

Trajectory = List[Transition]


Consider the following mdp with two states:

1. Orderly, or messy, room. Room starts off orderly. 
2. Tidying definitely results in being orderly. Not tidying gets a 0.3-chance of transitioning to messy. 
3. Tidying an orderly room and ignoring a messy room both gets $-1$. Ignoring a orderly room gets $1$, while tidying a messy room is $0$

In [10]:
tidy_mdp = SimpleMDP(
    S = 2, A = 2, 
    mu = jnp.array([1., 0.]),
    P = jnp.array([
    [
        [1., 0.], # Orderly, tidy -> orderly 
        [0.7, 0.3], # Orderly, ignore -> 0.3-chance messy 
    ], [
        [1., 0.], # Messy, tidy -> orderly 
        [0., 1.], # Messy, ignore -> messy 
    ]]), 
    r = jnp.array([
        [-1., 1], 
        [0, -1], 
    ]), 
    H = 7
)

In [11]:
always_tidy_policy = (
    jnp.zeros((7, 2, 2))
    .at[:, :, 0].set(1.) # No matter the time and state, always tidy 
)
messy_policy = (
    jnp.zeros((7, 2, 2))
    .at[:, :, 1].set(1.)
)
weekend_tidy_policy = (
    jnp.zeros((7, 2, 2))
    .at[5:7, :, 0].set(1.)
    .at[:5, :, 1].set(1.)
)

In [19]:
def log_likelihood(
    mdp: SimpleMDP, 
    policy: Float[Array, "S A"], 
    traj: Trajectory):
    # Initial state and initial action 
    result = jnp.log(
        mdp.mu[traj[0].s], 
    ) + jnp.log(
        policy[traj[0].s, traj[0].a]
    )

    for t in range(1, mdp.H):
        result += jnp.log(
            mdp.P[traj[t-1].s, traj[t-1].a, traj[t].s] * 
            policy[traj[t].s, traj[t].a]
        )
    return result 

The value function is defined as 
\[ 
    V^\pi_h(s) = \mathbb E_{\tau \sim \rho^\tau}[r_h + \dots + r_{H-1} \mid s_h = s]
\] 
The $Q$-function is defined as 
\[ 
    Q^\pi_h(s, a) = \mathbb E_{\tau \sim \rho^\tau}[r_h + \dots + r_{H-1} \mid s_h = s, a_h=a]
\] 